# Object_Detection Phase 5 — YOLOv11 파인튜닝

**목적**: AI Hub 음식이미지 데이터로 yolo11s.pt 파인튜닝 → 한식/지방특산물 직접 탐지

**대상 데이터셋**: AI Hub '비전영역, 음식이미지 및 정보소개 텍스트 데이터' (dataSetSn=71564)

**결과물**: `best.pt` → 로컬 `Object_Detection/models/korean_food_v1_best.pt` 에 저장

**현재 파이프라인 연동**: `config/detection_config.yaml` 모델 경로만 교체하면 완료

---
### 실행 순서
1. 런타임 → GPU(A100) 설정 확인
2. 셀을 순서대로 실행
3. Step 7 완료 후 Drive에서 `best.pt` 다운로드

## Step 0 — 환경 설정 및 Drive 마운트

In [ ]:
# GPU 확인
import torch
print('GPU 사용 가능:', torch.cuda.is_available())
print('GPU 이름:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
!nvidia-smi

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive'

# 작업 폴더 구조
DATASET_DIR   = f'{DRIVE_ROOT}/aihub_food'          # AI Hub 압축파일 위치
FINETUNE_DIR  = f'{DRIVE_ROOT}/finetune_dataset'     # 변환된 YOLO 데이터셋
RUNS_DIR      = f'{DRIVE_ROOT}/runs'                 # 학습 결과 저장

for d in [FINETUNE_DIR, RUNS_DIR,
          f'{FINETUNE_DIR}/train/images', f'{FINETUNE_DIR}/train/labels',
          f'{FINETUNE_DIR}/val/images',   f'{FINETUNE_DIR}/val/labels']:
    os.makedirs(d, exist_ok=True)

print('Drive 마운트 완료')
print(f'데이터셋 경로: {DATASET_DIR}')
print(f'학습 결과 경로: {RUNS_DIR}')

In [ ]:
# 패키지 설치
!pip install ultralytics -q
print('ultralytics 설치 완료')

## Step 1 — 타겟 클래스 정의

> `stt_keywords.yaml` 과 `clip_queries.yaml` 기준으로 우리 프로젝트에서 필요한 클래스만 정의.
> Phase 1에서 탐지 0건이었던 항목 우선.

In [ ]:
# ─────────────────────────────────────────────────────
# 우리 프로젝트 타겟 클래스 (stt_keywords.yaml + clip_queries.yaml 기준)
#
# Phase 1 탐지 0건 → 파인튜닝 대상 우선
# AI Hub 음식명과 일치 여부 확인 필요 → Step 2에서 실제 데이터 탐색 후 보완
# ─────────────────────────────────────────────────────

TARGET_CLASSES = [
    # 지방특산물 (stt_keywords.yaml 기준)
    '굴비',
    '조기',
    '대게',
    '전복',
    '장어',
    '한우',
    '흑돼지',
    '젓갈',
    '멸치',

    # 한식 (Phase 1 탐지 0건 + clip_queries.yaml 기준)
    '비빔밥',
    '김치찌개',
    '된장찌개',
    '삼겹살',
    '불고기',
    '갈비',
    '전골',
    '김치',

    # 과일채소 (stt_keywords.yaml 기준)
    '사과',
    '딸기',
    '수박',
    '감귤',
    '복숭아',
]

# class_id 매핑 (YOLO 라벨 파일에 사용)
CLASS_TO_ID = {cls: idx for idx, cls in enumerate(TARGET_CLASSES)}
ID_TO_CLASS = {idx: cls for cls, idx in CLASS_TO_ID.items()}

print(f'타겟 클래스 수: {len(TARGET_CLASSES)}')
for i, cls in enumerate(TARGET_CLASSES):
    print(f'  {i:2d}: {cls}')

## Step 2 — AI Hub 데이터 탐색

> AI Hub에서 다운로드한 압축파일을 Drive의 `aihub_food/` 폴더에 먼저 업로드할 것.
> 폴더 구조: A(특수외식) / B(일반외식·배달) 폴더만 필요. C, D는 불필요.
>
> **권장 다운로드 폴더**
> - `A/13_향토음식/` ← 굴비, 대게, 한우 등 지방특산물
> - `B/12_일반외식/` ← 비빔밥, 김치찌개 등 한식
> - `B/11_배달음식/` ← 삼겹살, 치킨 등

In [ ]:
import os, json
from pathlib import Path
from collections import defaultdict

# AI Hub 데이터 실제 구조 탐색
# → 우리 타겟 클래스명이 AI Hub에서 어떤 이름으로 있는지 확인

def explore_aihub_labels(dataset_dir: str, sample_limit: int = 3):
    """AI Hub JSON 라벨에서 실제 음식명 목록 수집"""
    food_names = defaultdict(int)
    json_files = list(Path(dataset_dir).rglob('*.json'))
    print(f'JSON 파일 수: {len(json_files)}')

    for jf in json_files[:500]:  # 샘플 500개만 탐색
        try:
            with open(jf, encoding='utf-8') as f:
                data = json.load(f)
            # AI Hub 라벨 구조: data.food_type.fc 또는 파일명에 음식명 포함
            food_type = data.get('data', {}).get('food_type', {})
            fc = food_type.get('fc', '')
            if fc:
                food_names[fc] += 1
        except:
            pass

    # 타겟 클래스와 매칭 확인
    print('\n=== 발견된 음식명 (상위 30개) ===')
    for name, cnt in sorted(food_names.items(), key=lambda x: -x[1])[:30]:
        match = '✅' if any(t in name or name in t for t in TARGET_CLASSES) else '  '
        print(f'{match} {name}: {cnt}건')

    return food_names

# Drive에 데이터 업로드 후 실행
if os.path.exists(DATASET_DIR):
    food_names = explore_aihub_labels(DATASET_DIR)
else:
    print(f'⚠️  {DATASET_DIR} 없음')
    print('AI Hub에서 데이터 다운로드 후 Drive에 업로드하세요')
    print('권장 폴더: A/13_향토음식/, B/12_일반외식/, B/11_배달음식/')

## Step 3 — AI Hub JSON → YOLO 포맷 변환

In [ ]:
import shutil
import random
from PIL import Image

# ─────────────────────────────────────────────────────
# AI Hub JSON 라벨 구조
# {
#   "data": {
#     "image_info": { "file_name": "xxx.jpg", "width": 1280, "height": 960 },
#     "2d_annotation": { "x": 100, "y": 200, "width": 300, "height": 250 },
#     "food_type": { "fc": "비빔밥", "loc": "한국" }
#   }
# }
#
# YOLO 포맷: class_id x_center y_center width height (0~1 정규화)
# ─────────────────────────────────────────────────────

def aihub_json_to_yolo(json_path: str, img_dir: str) -> tuple:
    """
    AI Hub JSON → YOLO 라벨 변환

    Returns:
        (class_id, yolo_label_str, img_path) or None
    """
    with open(json_path, encoding='utf-8') as f:
        data = json.load(f).get('data', {})

    # 음식명 추출
    food_name = data.get('food_type', {}).get('fc', '')

    # 타겟 클래스 매칭 (부분 일치 허용)
    matched_class = None
    for target in TARGET_CLASSES:
        if target in food_name or food_name in target:
            matched_class = target
            break
    if matched_class is None:
        return None

    # 이미지 정보
    img_info = data.get('image_info', {})
    img_w = img_info.get('width', 0)
    img_h = img_info.get('height', 0)
    file_name = img_info.get('file_name', '')

    if img_w == 0 or img_h == 0 or not file_name:
        return None

    # bbox 변환: AI Hub (x, y, w, h, 절대좌표 좌측상단) → YOLO (cx, cy, w, h, 정규화)
    ann = data.get('2d_annotation', {})
    x, y = ann.get('x', 0), ann.get('y', 0)
    bw, bh = ann.get('width', 0), ann.get('height', 0)

    if bw == 0 or bh == 0:
        return None

    x_center = (x + bw / 2) / img_w
    y_center = (y + bh / 2) / img_h
    w_norm   = bw / img_w
    h_norm   = bh / img_h

    # 범위 검증 (0~1)
    for val in [x_center, y_center, w_norm, h_norm]:
        if not (0.0 <= val <= 1.0):
            return None

    class_id = CLASS_TO_ID[matched_class]
    yolo_label = f'{class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}'

    # 이미지 파일 경로 탐색
    img_path = Path(img_dir) / file_name
    if not img_path.exists():
        # 재귀 탐색
        candidates = list(Path(img_dir).rglob(file_name))
        if not candidates:
            return None
        img_path = candidates[0]

    return class_id, yolo_label, str(img_path), matched_class


def build_yolo_dataset(
    aihub_dir: str,
    output_dir: str,
    val_ratio: float = 0.2,
    max_per_class: int = 500,  # 클래스당 최대 이미지 수
):
    """
    AI Hub 데이터 → YOLO 학습 데이터셋 구성
    train/val 분리 (8:2)
    """
    json_files = list(Path(aihub_dir).rglob('*.json'))
    print(f'JSON 파일 총계: {len(json_files):,}개')

    class_records = defaultdict(list)  # {class_name: [(yolo_label, img_path), ...]}
    skip_count = 0

    for jf in json_files:
        result = aihub_json_to_yolo(str(jf), str(Path(jf).parent))
        if result is None:
            skip_count += 1
            continue
        class_id, yolo_label, img_path, class_name = result
        class_records[class_name].append((yolo_label, img_path))

    print(f'\n=== 클래스별 수집 현황 ===')
    total_used = 0
    for cls in TARGET_CLASSES:
        cnt = len(class_records[cls])
        status = '✅' if cnt >= 100 else ('⚠️ ' if cnt > 0 else '❌')
        print(f'{status} {cls}: {cnt}장')
        total_used += min(cnt, max_per_class)
    print(f'\n스킵(타겟 아님): {skip_count:,}개')
    print(f'사용 예정 총계: {total_used:,}장')

    # train/val 분리 및 복사
    train_count = val_count = 0
    for class_name, records in class_records.items():
        if not records:
            continue
        # 클래스당 최대 수 제한
        if len(records) > max_per_class:
            records = random.sample(records, max_per_class)

        random.shuffle(records)
        split = int(len(records) * (1 - val_ratio))
        train_set = records[:split]
        val_set   = records[split:]

        for split_name, split_records in [('train', train_set), ('val', val_set)]:
            for yolo_label, img_path in split_records:
                img_path = Path(img_path)
                stem = img_path.stem

                dst_img = Path(output_dir) / split_name / 'images' / img_path.name
                dst_lbl = Path(output_dir) / split_name / 'labels' / f'{stem}.txt'

                shutil.copy2(str(img_path), str(dst_img))
                dst_lbl.write_text(yolo_label + '\n', encoding='utf-8')

                if split_name == 'train':
                    train_count += 1
                else:
                    val_count += 1

    print(f'\n=== 데이터셋 구성 완료 ===')
    print(f'train: {train_count:,}장')
    print(f'val:   {val_count:,}장')
    return train_count, val_count


# 실행
if os.path.exists(DATASET_DIR):
    train_count, val_count = build_yolo_dataset(
        aihub_dir=DATASET_DIR,
        output_dir=FINETUNE_DIR,
        val_ratio=0.2,
        max_per_class=500,
    )
else:
    print(f'⚠️  {DATASET_DIR} 없음 — AI Hub 데이터 업로드 후 재실행')

## Step 4 — data.yaml 생성

In [ ]:
import yaml

# YOLO 학습용 data.yaml 생성
# 우리 프로젝트 TARGET_CLASSES 순서와 동일하게 작성

data_yaml = {
    'train': f'{FINETUNE_DIR}/train/images',
    'val':   f'{FINETUNE_DIR}/val/images',
    'nc':    len(TARGET_CLASSES),
    'names': TARGET_CLASSES,
}

yaml_path = f'{FINETUNE_DIR}/data.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_yaml, f, allow_unicode=True, default_flow_style=False)

print('data.yaml 생성 완료:')
print(f'  경로: {yaml_path}')
print(f'  클래스 수: {len(TARGET_CLASSES)}')
print(f'  클래스 목록:')
for i, cls in enumerate(TARGET_CLASSES):
    print(f'    {i}: {cls}')

## Step 5 — 파인튜닝 학습

> `yolo11s.pt` → 전이학습 (Transfer Learning)
> 결과: `{RUNS_DIR}/korean_food_v1/weights/best.pt`

In [ ]:
from ultralytics import YOLO

# 현재 프로젝트와 동일한 base 모델: yolo11s.pt
# (detector.py 기본값 model_name='yolo11s.pt')
model = YOLO('yolo11s.pt')

print('=== 학습 시작 ===')
print(f'base 모델: yolo11s.pt')
print(f'data.yaml: {FINETUNE_DIR}/data.yaml')
print(f'결과 저장: {RUNS_DIR}/korean_food_v1/')

results = model.train(
    data=f'{FINETUNE_DIR}/data.yaml',

    # ── 학습 설정 ──
    epochs=100,          # 전체 반복 횟수
    imgsz=640,           # 입력 이미지 크기 (현재 파이프라인 기본값과 동일)
    batch=16,            # A100 기준 최적값
    device=0,            # GPU 사용

    # ── 조기 종료 ──
    patience=20,         # 20 epoch 성능 개선 없으면 자동 종료

    # ── 저장 경로 ──
    project=RUNS_DIR,
    name='korean_food_v1',
    exist_ok=True,

    # ── 현재 프로젝트 설정 맞춤 ──
    # detection_config.yaml의 inference 설정과 동일
    conf=0.5,            # confidence_threshold
    iou=0.45,            # iou_threshold (NMS)
    max_det=100,         # max_det

    # ── 로깅 ──
    verbose=True,
)

print('\n=== 학습 완료 ===')
print(f'best.pt 저장 위치: {RUNS_DIR}/korean_food_v1/weights/best.pt')

## Step 6 — 검증 및 성능 확인

In [ ]:
from ultralytics import YOLO
import pandas as pd

# best.pt로 검증
best_pt = f'{RUNS_DIR}/korean_food_v1/weights/best.pt'
model_ft = YOLO(best_pt)

metrics = model_ft.val(data=f'{FINETUNE_DIR}/data.yaml', device=0)

print('\n=== 검증 결과 ===')
print(f'mAP@0.5:      {metrics.box.map50:.4f}  (목표: ≥ 0.60)')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f}')
print(f'Precision:    {metrics.box.mp:.4f}')
print(f'Recall:       {metrics.box.mr:.4f}')

# 클래스별 AP 출력
print('\n=== 클래스별 AP@0.5 ===')
if hasattr(metrics.box, 'ap_class_index'):
    rows = []
    for i, class_idx in enumerate(metrics.box.ap_class_index):
        cls_name = TARGET_CLASSES[int(class_idx)] if int(class_idx) < len(TARGET_CLASSES) else str(class_idx)
        ap50 = metrics.box.ap50[i] if hasattr(metrics.box, 'ap50') else 0
        rows.append({'클래스': cls_name, 'AP@0.5': round(float(ap50), 4)})
    df = pd.DataFrame(rows).sort_values('AP@0.5', ascending=False)
    print(df.to_string(index=False))

# 합격 기준 판정
map50 = metrics.box.map50
if map50 >= 0.60:
    print(f'\n✅ mAP@0.5 = {map50:.4f} → 합격 (≥ 0.60)')
    print('→ best.pt 로컬에 다운로드하여 Object_Detection/models/ 에 저장하세요')
else:
    print(f'\n⚠️  mAP@0.5 = {map50:.4f} → 미달 (< 0.60)')
    print('→ 데이터 추가 또는 epochs 늘려서 재학습 권장')

## Step 7 — A/B 비교 (기존 yolo11s.pt vs 파인튜닝 best.pt)

In [ ]:
from ultralytics import YOLO
import glob, cv2, numpy as np

# val 이미지 샘플 10장으로 빠른 A/B 비교
val_images = glob.glob(f'{FINETUNE_DIR}/val/images/*.jpg')[:10]

if not val_images:
    print('val 이미지 없음 — Step 3 완료 후 실행하세요')
else:
    model_base = YOLO('yolo11s.pt')                              # 기존 모델
    model_ft   = YOLO(f'{RUNS_DIR}/korean_food_v1/weights/best.pt')  # 파인튜닝 모델

    base_korean = 0
    ft_korean   = 0

    KOREAN_LABELS = set(TARGET_CLASSES)  # 우리 타겟 클래스

    for img_path in val_images:
        # 기존 모델 추론
        preds_base = model_base(img_path, conf=0.5, verbose=False)
        for pred in preds_base:
            for box in pred.boxes:
                label = pred.names[int(box.cls)]
                if label in KOREAN_LABELS:
                    base_korean += 1

        # 파인튜닝 모델 추론
        preds_ft = model_ft(img_path, conf=0.5, verbose=False)
        for pred in preds_ft:
            for box in pred.boxes:
                label = pred.names[int(box.cls)]
                if label in KOREAN_LABELS:
                    ft_korean += 1

    print('=== A/B 비교 결과 (val 샘플 10장) ===')
    print(f'기존 yolo11s.pt     한식 탐지: {base_korean}건')
    print(f'파인튜닝 best.pt    한식 탐지: {ft_korean}건')
    improvement = ft_korean - base_korean
    print(f'개선: +{improvement}건')

    if ft_korean > base_korean:
        print('\n✅ 파인튜닝 효과 확인 — 로컬 적용 진행하세요')
    else:
        print('\n⚠️  효과 미미 — 데이터셋 품질 및 클래스 매칭 재확인 필요')

## Step 8 — best.pt 다운로드 및 로컬 적용 안내

In [ ]:
import os
from pathlib import Path

best_pt_path = f'{RUNS_DIR}/korean_food_v1/weights/best.pt'

if os.path.exists(best_pt_path):
    size_mb = os.path.getsize(best_pt_path) / 1024 / 1024
    print(f'✅ best.pt 확인')
    print(f'   경로: {best_pt_path}')
    print(f'   크기: {size_mb:.1f} MB')
    print()
    print('=== 로컬 적용 방법 ===')
    print()
    print('1. Drive에서 best.pt 다운로드')
    print(f'   Drive 경로: 내 드라이브/runs/korean_food_v1/weights/best.pt')
    print()
    print('2. 로컬 프로젝트에 저장')
    print('   Object_Detection/models/korean_food_v1_best.pt')
    print()
    print('3. config/detection_config.yaml 수정 (한 줄만)')
    print('   변경 전: name: yolo11s.pt')
    print('   변경 후: name: models/korean_food_v1_best.pt')
    print()
    print('4. 로컬에서 검증')
    print('   python scripts/batch_detect.py --input-dir data/trailers_아름 --limit 10 --random')
    print('   → 한식 라벨(비빔밥, 굴비 등) 탐지 건수 확인')
    print()
    print('5. 기존 테스트 통과 확인')
    print('   python -m pytest tests/test_phase1_setup.py -v  # 13/13 PASS 유지')
    print()
    print('⚠️  주의: models/*.pt 는 .gitignore 대상 — 커밋하지 말 것')
    print('   팀원 공유: Drive 링크 또는 직접 전달')
else:
    print(f'⚠️  best.pt 없음: {best_pt_path}')
    print('Step 5 학습 완료 후 재실행하세요')

---

## 참고 — 현재 프로젝트 파이프라인 연동 포인트

```
Object_Detection/
├── src/detector.py          ← Detector(model_name=...) — 모델 경로만 바뀜
├── config/detection_config.yaml  ← name: models/korean_food_v1_best.pt 로 교체
├── models/
│   └── korean_food_v1_best.pt   ← Drive에서 다운로드한 파일 (gitignore 대상)
└── data/
    └── vod_detected_object.parquet  ← 한식 라벨 포함 재생성
```

### parquet 스키마는 변경 없음

| 컬럼 | 타입 | 비고 |
|------|------|------|
| `vod_id` | str | 동일 |
| `frame_ts` | float | 동일 |
| `label` | str | 한식 라벨 추가됨 (비빔밥, 굴비 등) |
| `confidence` | float | 동일 |
| `bbox` | list[float] | 동일 |

Shopping_Ad 연동 변경 없이 `label` 다양성만 증가.